## This file contains the test of the out of the box CNN on our test set

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import csv
from pathlib import Path
import torch
from PIL import Image as PILImage
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from transformers import AutoImageProcessor, AutoModelForImageClassification

In [ ]:
ROOT_DATASET = "/content/drive/MyDrive/Project 1-1/Dataset/Images/Test"
MODEL_IDENTIFIER = "Ateeqq/ai-vs-human-image-detector"
VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
PARENT_TO_LABEL = {
    "ai": 1,
    "not_ai": 0,
}

Using device: cpu


In [ ]:
processor = AutoImageProcessor.from_pretrained(MODEL_IDENTIFIER)
model = AutoModelForImageClassification.from_pretrained(MODEL_IDENTIFIER).to(device)
model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/372M [00:00<?, ?B/s]

SiglipForImageClassification(
  (vision_model): SiglipVisionTransformer(
    (embeddings): SiglipVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
      (position_embedding): Embedding(196, 768)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-11): 12 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): SiglipMLP(
            (activation_fn): GELUTanh()
            (fc1): Linear(in_features=768, out_features=30

In [ ]:
def get_label_from_path(path):
    folder = Path(path).name.strip().lower()
    return PARENT_TO_LABEL.get(folder, None)


In [ ]:
y_true = []
y_pred = []

print(f"Scanning dataset: {ROOT_DATASET}\n")

for root, dirs, files in os.walk(ROOT_DATASET):
    label = get_label_from_path(root)

    if label is None:
        continue

    for filename in sorted(files):
        image_path = os.path.join(root, filename)

        ext = os.path.splitext(filename)[1].lower()
        if ext not in VALID_EXT:
            print(f"[SKIP FILE - EXTENSION] {image_path}")
            continue

        try:
            with PILImage.open(image_path) as img:
                image = img.convert("RGB")
        except Exception as e:
            print(f"[SKIP FILE - OPEN ERROR] {image_path} | {e}")
            continue

        try:
            inputs = processor(images=image, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
        except Exception as e:
            print(f"[SKIP FILE - PREPROCESS ERROR] {image_path} | {e}")
            continue

        try:
            with torch.no_grad():
                outputs = model(**inputs)
                logits = outputs.logits
        except Exception as e:
            print(f"[SKIP FILE - INFERENCE ERROR] {image_path} | {e}")
            continue


        pred_idx = int(logits.argmax(-1).item())

        id2label = getattr(model.config, "id2label", None) or {}
        pred_label_name = id2label.get(pred_idx, f"label_{pred_idx}")

        # Map prediction to binary label
        name_lc = pred_label_name.lower()
        if name_lc.startswith("hum") or "human" in name_lc or "real" in name_lc:
            pred_label = 0
        elif name_lc.startswith("ai") or "generated" in name_lc or "synthetic" in name_lc:
            pred_label = 1
        else:
            if logits.shape[-1] == 2:
                pred_label = int(pred_idx)

        y_true.append(int(label))
        y_pred.append(pred_label)


Scanning dataset: /content/drive/MyDrive/Project 1-1/Dataset/Images/Test



In [ ]:
print("\n=== METRICS ===")

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")

cm = confusion_matrix(y_true, y_pred, labels=[1,0])
print("\nConfusion Matrix (rows=true [ai,human], cols=pred [ai,human]):")
print(cm)


=== METRICS ===
Accuracy:  0.5000
Precision: 0.4390
Recall:    0.9000
F1 Score:  0.5902

Confusion Matrix (rows=true [ai,human], cols=pred [ai,human]):
[[18  2]
 [23  7]]


In [ ]:
ai_count = y_true.count(1)
not_ai_count = y_true.count(0)

print(f"\nClass Balance (True Labels):")
print(f"  AI instances: {ai_count}")
print(f"  Not AI instances: {not_ai_count}")


Class Balance (True Labels):
  AI instances: 20
  Not AI instances: 30


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

print("\n=== Displaying Example Images ===")
for i, example in enumerate(print_examples):
    image_path = example['image_path']
    true_name = 'AI' if example['true_label'] == 1 else 'Human'
    pred_name = 'AI' if example['predicted_label'] == 1 else 'Human'

    try:
        img = Image.open(image_path)

        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.title(f"Example {i+1}: True: {true_name}, Predicted: {pred_name}\nImage: {image_path.split('/')[-1]}")
        plt.axis('off')
        plt.show()

    except Exception as e:
        print(f"Could not display image {image_path}: {e}")
    print("-------------------------------------------")

Output hidden; open in https://colab.research.google.com to view.